In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, norm
import statsmodels.stats.proportion as smp
import warnings
warnings.filterwarnings('ignore')

BASE = r'C:\Users\WELCOME\Desktop\DataAnalysis_Projects\ab-testing-marketing-analysis'

print("All libraries loaded successfully")

In [ ]:
df = pd.read_csv(BASE + r'\data\marketing_AB.csv')

ad  = df[df['test group'] == 'ad']
psa = df[df['test group'] == 'psa']

n_ad  = len(ad)
n_psa = len(psa)

conv_ad  = ad['converted'].sum()
conv_psa = psa['converted'].sum()

rate_ad  = ad['converted'].mean()
rate_psa = psa['converted'].mean()

print(f"Ad  group : {n_ad:,} users, {conv_ad:,} conversions, {rate_ad*100:.4f}% rate")
print(f"PSA group : {n_psa:,} users, {conv_psa:,} conversions, {rate_psa*100:.4f}% rate")
print(f"\nAbsolute uplift  : {(rate_ad - rate_psa)*100:.4f} percentage points")
print(f"Relative uplift  : {((rate_ad - rate_psa)/rate_psa)*100:.2f}%")

In [ ]:
# Check if the test had enough power BEFORE looking at results
# Standard: alpha=0.05, power=0.80, MDE=20% relative lift

from statsmodels.stats.power import NormalIndPower

alpha      = 0.05
power      = 0.80
baseline   = rate_psa                    # control group rate
mde        = 0.20                        # minimum detectable effect (20% relative)
effect     = baseline * mde              # absolute effect size
p1         = baseline
p2         = baseline + effect

# Required sample size per group
analysis   = NormalIndPower()
required_n = analysis.solve_power(
    effect_size = (p2 - p1) / np.sqrt((p1*(1-p1) + p2*(1-p2))/2),
    alpha       = alpha,
    power       = power,
    alternative = 'two-sided'
)

print("Sample Size and Power Analysis:")
print(f"  Baseline conversion rate (PSA) : {baseline*100:.4f}%")
print(f"  Minimum detectable effect      : {mde*100:.0f}% relative lift")
print(f"  Required sample size per group : {required_n:,.0f}")
print(f"  Actual Ad group size           : {n_ad:,}")
print(f"  Actual PSA group size          : {n_psa:,}")
print(f"\n  Ad group powered               : {'YES' if n_ad >= required_n else 'NO'}")
print(f"  PSA group powered              : {'YES' if n_psa >= required_n else 'NO'}")

In [ ]:
# Build contingency table
#                  Converted    Not Converted
# Ad (Test)        conv_ad      n_ad - conv_ad
# PSA (Control)    conv_psa     n_psa - conv_psa

contingency = np.array([
    [conv_ad,  n_ad  - conv_ad],
    [conv_psa, n_psa - conv_psa]
])

chi2, p_chi2, dof, expected = chi2_contingency(contingency)

print("Chi-Square Test of Independence:")
print(f"\nContingency Table:")
print(f"                Converted    Not Converted    Total")
print(f"  Ad (Test)     {conv_ad:>9,}    {n_ad-conv_ad:>13,}    {n_ad:>6,}")
print(f"  PSA (Control) {conv_psa:>9,}    {n_psa-conv_psa:>13,}    {n_psa:>6,}")
print(f"\nChi-square statistic : {chi2:.4f}")
print(f"Degrees of freedom   : {dof}")
print(f"P-value              : {p_chi2:.6f}")
print(f"\nResult: {'REJECT null hypothesis' if p_chi2 < 0.05 else 'FAIL TO REJECT null hypothesis'}")
print(f"Conclusion: The difference {'IS' if p_chi2 < 0.05 else 'IS NOT'} statistically significant at alpha=0.05")

In [ ]:
# Two-proportion z-test
count  = np.array([conv_ad, conv_psa])
nobs   = np.array([n_ad, n_psa])

z_stat, p_ztest = smp.proportions_ztest(count, nobs, alternative='two-sided')

# Confidence interval for the difference
diff   = rate_ad - rate_psa
se     = np.sqrt(rate_ad*(1-rate_ad)/n_ad + rate_psa*(1-rate_psa)/n_psa)
ci_low = diff - 1.96 * se
ci_hi  = diff + 1.96 * se

print("Two-Proportion Z-Test:")
print(f"\nZ-statistic          : {z_stat:.4f}")
print(f"P-value              : {p_ztest:.6f}")
print(f"\n95% Confidence Interval for difference in conversion rates:")
print(f"  [{ci_low*100:.4f}%, {ci_hi*100:.4f}%]")
print(f"\nInterpretation:")
print(f"  We are 95% confident the true difference in conversion rates")
print(f"  between Ad and PSA groups is between {ci_low*100:.4f}% and {ci_hi*100:.4f}%")
print(f"\nResult: {'REJECT null hypothesis' if p_ztest < 0.05 else 'FAIL TO REJECT null hypothesis'}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

x = np.linspace(-5, 5, 1000)
y = norm.pdf(x)

ax.plot(x, y, color='#333333', linewidth=2)
ax.fill_between(x, y, where=(x < -1.96), color='#D85A30', alpha=0.4, label='Rejection region (α=0.05)')
ax.fill_between(x, y, where=(x > 1.96),  color='#D85A30', alpha=0.4)
ax.fill_between(x, y, where=((-1.96 <= x) & (x <= 1.96)), color='#378ADD', alpha=0.15, label='Acceptance region')

ax.axvline(z_stat, color='#1D9E75', linewidth=2.5, linestyle='--', label=f'Z-statistic = {z_stat:.2f}')
ax.axvline(-1.96,  color='#D85A30', linewidth=1.5, linestyle=':')
ax.axvline(1.96,   color='#D85A30', linewidth=1.5, linestyle=':')

ax.text(z_stat + 0.1, 0.35, f'Z = {z_stat:.2f}\np = {p_ztest:.4f}',
        fontsize=10, color='#1D9E75', fontweight='bold')
ax.text(-1.96, 0.42, 'Critical\nvalue\n-1.96', fontsize=8, ha='center', color='#D85A30')
ax.text( 1.96, 0.42, 'Critical\nvalue\n+1.96', fontsize=8, ha='center', color='#D85A30')

ax.set_title('Z-Test Result — Standard Normal Distribution', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Z-score', fontsize=11)
ax.set_ylabel('Probability Density', fontsize=11)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(BASE + r'\data\ztest_result.png', dpi=150, bbox_inches='tight')
plt.show()
print("Z-test chart saved.")

In [ ]:
# Cohen's h for proportions
h = 2 * np.arcsin(np.sqrt(rate_ad)) - 2 * np.arcsin(np.sqrt(rate_psa))

print("Effect Size and Practical Significance:")
print(f"\nCohen's h            : {h:.4f}")
print(f"Interpretation       : {'Small' if abs(h) < 0.2 else 'Medium' if abs(h) < 0.5 else 'Large'} effect")
print(f"\nAbsolute uplift      : {(rate_ad-rate_psa)*100:.4f} percentage points")
print(f"Relative uplift      : {((rate_ad-rate_psa)/rate_psa)*100:.2f}%")
print(f"\nBusiness impact estimate:")
extra_conv_rate = rate_ad - rate_psa
users_per_month = 50000
extra_conversions = users_per_month * extra_conv_rate
print(f"  If we serve ads to {users_per_month:,} users per month:")
print(f"  Expected extra conversions : {extra_conversions:.0f}")
print(f"  (vs showing PSA instead)")

In [ ]:
# Did the ad work better on certain days?
seg_day = df.groupby(['most ads day', 'test group'])['converted'].agg(['sum','count','mean']).reset_index()
seg_day.columns = ['Day', 'Group', 'Conversions', 'Users', 'Rate']
seg_day['Rate %'] = (seg_day['Rate'] * 100).round(4)

dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
seg_day['Day'] = pd.Categorical(seg_day['Day'], categories=dow_order, ordered=True)
seg_day = seg_day.sort_values('Day')

# Uplift per day
ad_day  = seg_day[seg_day['Group']=='ad'][['Day','Rate %']].set_index('Day')
psa_day = seg_day[seg_day['Group']=='psa'][['Day','Rate %']].set_index('Day')
uplift  = (ad_day - psa_day).reset_index()
uplift.columns = ['Day', 'Uplift (pp)']

print("Conversion Rate Uplift by Day (Ad vs PSA):")
print(uplift.to_string(index=False))
print(f"\nBest day for ad uplift  : {uplift.loc[uplift['Uplift (pp)'].idxmax(), 'Day']}")
print(f"Worst day for ad uplift : {uplift.loc[uplift['Uplift (pp)'].idxmin(), 'Day']}")

In [ ]:
results = {
    'Metric'     : [
        'Ad Conversion Rate %',
        'PSA Conversion Rate %',
        'Absolute Uplift (pp)',
        'Relative Uplift %',
        'Chi-Square Statistic',
        'Chi-Square P-Value',
        'Z-Statistic',
        'Z-Test P-Value',
        'CI Lower 95%',
        'CI Upper 95%',
        "Cohen's h",
        'Effect Size',
        'Statistically Significant',
        'Recommendation'
    ],
    'Value'      : [
        round(rate_ad*100, 4),
        round(rate_psa*100, 4),
        round((rate_ad-rate_psa)*100, 4),
        round(((rate_ad-rate_psa)/rate_psa)*100, 2),
        round(chi2, 4),
        round(p_chi2, 6),
        round(z_stat, 4),
        round(p_ztest, 6),
        round(ci_low*100, 4),
        round(ci_hi*100, 4),
        round(h, 4),
        'Small' if abs(h) < 0.2 else 'Medium' if abs(h) < 0.5 else 'Large',
        'YES' if p_ztest < 0.05 else 'NO',
        'Deploy ad campaign' if p_ztest < 0.05 else 'Do not deploy — insufficient evidence'
    ]
}

results_df = pd.DataFrame(results)
results_df.to_csv(BASE + r'\data\hypothesis_test_results.csv', index=False)

print("Hypothesis Test Results:")
print(results_df.to_string(index=False))
print("\nResults saved.")